In [0]:
#Read Tables
import pyspark.sql.functions as F

trucks = spark.table("workspace.logistics_project.trucks")
truck_utilization = spark.table("workspace.logistics_project.truck_utilization_metrics")

In [0]:
#Join Tables
fleet_utilization = (
    truck_utilization
    .join(trucks, "truck_id", "inner")
)

In [0]:
#Calculate Fleet KPIs
fleet_metrics = (
    fleet_utilization
    .groupBy(
        "truck_id",
        "unit_number",
        "make",
        "model_year",
        "fuel_type",
        "status"
    )
    .agg(
        F.sum("trips_completed").alias("total_trips"),
        F.sum("total_miles").alias("total_miles"),
        F.round(F.sum("total_revenue"),2).alias("total_revenue"),
        F.round(F.avg("average_mpg"),2).alias("avg_mpg"),
        F.sum("maintenance_events").alias("maintenance_events"),
        F.round(F.sum("maintenance_cost"),2).alias("maintenance_cost"),
        F.round(F.sum("downtime_hours"),2).alias("downtime_hours"),
        F.round(F.avg("utilization_rate"),2).alias("utilization_rate")
    )
)

In [0]:
#Revenue Per Mile
fleet_metrics = fleet_metrics.withColumn(
    "revenue_per_mile",
    F.round(
        F.col("total_revenue") /
        F.col("total_miles"),
        2
    )
)

In [0]:
#View Results
display(
    fleet_metrics.orderBy(
        F.col("utilization_rate").desc()
    )
)

truck_id,unit_number,make,model_year,fuel_type,status,total_trips,total_miles,total_revenue,avg_mpg,maintenance_events,maintenance_cost,downtime_hours,utilization_rate,revenue_per_mile
TRK00044,8170,Volvo,2015,Diesel,Active,971,1403922,3045707.57,6.5,38,69825.23,1080.4,0.89,2.17
TRK00055,6070,Peterbilt,2017,Diesel,Active,977,1417530,2997124.44,6.48,29,60433.32,689.5,0.89,2.11
TRK00039,7172,International,2015,Diesel,Active,966,1366732,2904283.89,6.51,24,47581.13,557.0,0.88,2.12
TRK00015,2531,International,2015,Diesel,Active,961,1386604,2979377.79,6.5,21,33711.95,562.5,0.88,2.15
TRK00041,3658,International,2015,Diesel,Active,965,1404001,3017320.99,6.51,25,59685.24,524.7,0.88,2.15
TRK00053,9749,Peterbilt,2015,Diesel,Active,964,1378108,2990269.38,6.48,27,50297.23,635.0,0.88,2.17
TRK00025,8967,Freightliner,2018,Diesel,Active,963,1370297,2942525.23,6.48,23,44082.12,570.1,0.88,2.15
TRK00050,5668,Peterbilt,2015,Diesel,Active,956,1338947,2907686.62,6.5,29,55990.68,687.7,0.87,2.17
TRK00001,3463,Peterbilt,2016,Diesel,Active,951,1356397,2896338.17,6.49,24,51775.99,713.2,0.87,2.14
TRK00057,4178,Mack,2015,Diesel,Active,951,1377066,2933620.12,6.55,25,57753.94,651.2,0.87,2.13


In [0]:
display(fleet_metrics)

truck_id,unit_number,make,model_year,fuel_type,status,total_trips,total_miles,total_revenue,avg_mpg,maintenance_events,maintenance_cost,downtime_hours,utilization_rate,revenue_per_mile
TRK00011,7171,Freightliner,2015,Diesel,Active,927,1304183,2814044.12,6.52,23,46464.63,656.6,0.85,2.16
TRK00040,3527,International,2015,Diesel,Active,860,1178515,2553800.68,6.5,26,67557.74,665.1,0.78,2.17
TRK00050,5668,Peterbilt,2015,Diesel,Active,956,1338947,2907686.62,6.5,29,55990.68,687.7,0.87,2.17
TRK00051,8355,Mack,2016,Diesel,Active,912,1312973,2795182.65,6.53,23,49991.87,577.3,0.83,2.13
TRK00058,7026,Mack,2015,Diesel,Active,898,1299089,2809911.6,6.48,20,45745.44,414.9,0.82,2.16
TRK00059,8218,Mack,2015,Diesel,Active,860,1216720,2595479.77,6.56,18,34580.8,450.0,0.79,2.13
TRK00063,9239,Peterbilt,2015,Diesel,Active,869,1229384,2625773.77,6.49,27,46669.04,674.8,0.79,2.14
TRK00069,5096,Mack,2016,Diesel,Active,935,1318110,2843417.74,6.51,18,35126.99,366.7,0.85,2.16
TRK00070,1854,Kenworth,2015,Diesel,Active,943,1319519,2800116.98,6.52,15,36142.32,307.7,0.86,2.12
TRK00092,9224,Volvo,2015,Diesel,Active,902,1324217,2805193.13,6.46,24,41671.12,433.6,0.82,2.12


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
fleet_metrics.write \
    .format("delta") \
    .saveAsTable(
        "workspace.logistics_gold.fleet_utilization"
    )

In [0]:
#Create Gold Table
spark.sql("""
SHOW TABLES IN workspace.logistics_gold
""").show(truncate=False)

+--------------+-------------------+-----------+
|database      |tableName          |isTemporary|
+--------------+-------------------+-----------+
|logistics_gold|driver_performance |false      |
|logistics_gold|fleet_utilization  |false      |
|logistics_gold|route_profitability|false      |
+--------------+-------------------+-----------+



In [0]:
#MERGE
from delta.tables import DeltaTable

gold_table = DeltaTable.forName(
    spark,
    "workspace.logistics_gold.fleet_utilization"
)

gold_table.alias("target").merge(
    fleet_metrics.alias("source"),
    "target.truck_id = source.truck_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
fleet_metrics.show(5)

+--------+-----------+-------------+----------+---------+------+-----------+-----------+-------------+-------+------------------+----------------+--------------+----------------+----------------+
|truck_id|unit_number|         make|model_year|fuel_type|status|total_trips|total_miles|total_revenue|avg_mpg|maintenance_events|maintenance_cost|downtime_hours|utilization_rate|revenue_per_mile|
+--------+-----------+-------------+----------+---------+------+-----------+-----------+-------------+-------+------------------+----------------+--------------+----------------+----------------+
|TRK00011|       7171| Freightliner|      2015|   Diesel|Active|        927|    1304183|   2814044.12|   6.52|                23|        46464.63|         656.6|            0.85|            2.16|
|TRK00040|       3527|International|      2015|   Diesel|Active|        860|    1178515|   2553800.68|    6.5|                26|        67557.74|         665.1|            0.78|            2.17|
|TRK00050|       566

In [0]:
display(
    spark.table("workspace.logistics_gold.fleet_utilization")
)

truck_id,unit_number,make,model_year,fuel_type,status,total_trips,total_miles,total_revenue,avg_mpg,maintenance_events,maintenance_cost,downtime_hours,utilization_rate,revenue_per_mile
TRK00011,7171,Freightliner,2015,Diesel,Active,927,1304183,2814044.12,6.52,23,46464.63,656.6,0.85,2.16
TRK00040,3527,International,2015,Diesel,Active,860,1178515,2553800.68,6.5,26,67557.74,665.1,0.78,2.17
TRK00050,5668,Peterbilt,2015,Diesel,Active,956,1338947,2907686.62,6.5,29,55990.68,687.7,0.87,2.17
TRK00051,8355,Mack,2016,Diesel,Active,912,1312973,2795182.65,6.53,23,49991.87,577.3,0.83,2.13
TRK00058,7026,Mack,2015,Diesel,Active,898,1299089,2809911.6,6.48,20,45745.44,414.9,0.82,2.16
TRK00059,8218,Mack,2015,Diesel,Active,860,1216720,2595479.77,6.56,18,34580.8,450.0,0.79,2.13
TRK00063,9239,Peterbilt,2015,Diesel,Active,869,1229384,2625773.77,6.49,27,46669.04,674.8,0.79,2.14
TRK00069,5096,Mack,2016,Diesel,Active,935,1318110,2843417.74,6.51,18,35126.99,366.7,0.85,2.16
TRK00070,1854,Kenworth,2015,Diesel,Active,943,1319519,2800116.98,6.52,15,36142.32,307.7,0.86,2.12
TRK00092,9224,Volvo,2015,Diesel,Active,902,1324217,2805193.13,6.46,24,41671.12,433.6,0.82,2.12
